In [514]:
import numpy as np
import pandas as pd

In [515]:
def calculate_rate_constant(temperature: float, a: float, ea: float) -> float:
    """
    Function to calculate rate constant
    :param temperature: temperature in Kelvin
    :param a: pre-exponential factor in 1/s
    :param ea: activation energy in J/mol
    :return: reaction rate constant k in 1/s
    """
    return a * np.exp(-ea / (8.314 * temperature))

In [516]:
def calculate_conversion(k: float, time: float):
    """
    Function to calculate conversion
    :param k: reaction rate constant in 1/s
    :param time: reaction time in seconds
    :return: conversion as a fraction
    """
    return 1 - np.exp(-k * time)

In [517]:
def calculate_batch_time(k: float, target_conversion: float) -> float:
    """
    Function to calculate batch time
    :param k: reaction rate constant in 1/s
    :param target_conversion: desired conversion as a fraction
    :return: batch time in seconds
    """
    return -np.log(1 - target_conversion) / k

In [518]:
def analyze_reactor_performance(reactor_specs: dict[str, float]) -> dict[str, float]:
    """
    Function to analyze reactor performance
    :param reactor_specs: dictionary containing 'temperature', 'A', and 'Ea'
    :return: dict with analysis results
    """
    temperature = reactor_specs['temperature']
    a = reactor_specs['A']
    ea = reactor_specs['Ea']
    k = calculate_rate_constant(temperature, a, ea)

    time = reactor_specs['batch_time']
    conversion  = calculate_conversion(k, time)

    target_conversion = reactor_specs['target_conversion']
    required_time = calculate_batch_time(k, target_conversion)

    return {
        'k': k,
        'actual_conversion': conversion,
        'required_time': required_time,
        'conversion_percent': conversion * 100,
        'time_hours': time / 3600
    }

In [519]:
df = pd.DataFrame()
df['Pharmaceutical synthesis'] = [320, 2e10, 85000, 7200, 0.9]
df['Specialty chemical'] = [330, 3e8, 72000, 5400, 0.85]
df['Industrial bulk chemical'] = [340, 1.2e10, 78000, 3600, 0.95]

df.loc[len(df)] = [calculate_rate_constant(df[j][0], df[j][1], df[j][2]) for i, j in enumerate(df)]
df.loc[len(df)] = [calculate_conversion(df[j][5], df[j][3]) for i, j in enumerate(df)]
df.loc[len(df)] = [calculate_batch_time(df[j][5], df[j][4]) for i, j in enumerate(df)]
df.loc[len(df)] = [df[j][7] <= df[j][3] for i, j in enumerate(df)]

df.index = ['Temperature (K)', 'Pre-exponential factor (A) (1/s)', 'Activation energy (Ea) (J/mol)', 'Batch time (s)', 'Target conversion', 'Reaction rate (k)', 'Actual conversion', 'Time required for target (s)', 'Sufficient batch time?']

rows = list(df.index)

df = (df.style
      .format('{:,.0f}', pd.IndexSlice[:, :])
      .format(lambda x: f'{x*100:.0f}%', pd.IndexSlice[rows[4:7:2], :])
      .format('{:.0e}', pd.IndexSlice[rows[1], :])
      .format('{:.5f}', pd.IndexSlice[rows[5], :])
      .format(lambda x: bool(x), pd.IndexSlice[rows[8], :])
      )

df

,Pharmaceutical synthesis,Specialty chemical,Industrial bulk chemical
Temperature (K),320,330,340
Pre-exponential factor (A) (1/s),2e+10,3e+08,1e+10
Activation energy (Ea) (J/mol),"85,000","72,000","78,000"
Batch time (s),"7,200","5,400","3,600"
Target conversion,90%,85%,95%
Reaction rate (k),0.00027,0.00120,0.01246
Actual conversion,85%,100%,100%
Time required for target (s),"8,640","1,578",240
Sufficient batch time?,False,True,True
